In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

# ----------------------------
# Helper functions
# ----------------------------

def safe_div(a, b):
    return np.nan if b in [0, None, np.nan] else a / b

def compute_roic(info):
    nopat = info.get("ebitda", np.nan) * (1 - info.get("taxRate", 0.25))
    invested_capital = (
        info.get("totalDebt", 0)
        + info.get("totalEquity", 0)
        - info.get("cash", 0)
    )
    return safe_div(nopat, invested_capital)

def verdict(score):
    if score >= 75:
        return "🔥 Strong – Dig deeper"
    elif score >= 60:
        return "🧐 Interesting – Worth analysis"
    elif score >= 45:
        return "😐 Neutral – Needs catalyst"
    else:
        return "❌ Weak – Pass for now"

# ----------------------------
# Main Screener
# ----------------------------

def screen_stock(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info

    # --- Core metrics ---
    price = info.get("currentPrice")
    pe = info.get("trailingPE")
    eps_growth = info.get("earningsQuarterlyGrowth", 0) * 100
    rev_growth = info.get("revenueGrowth", 0) * 100
    div_yield = info.get("dividendYield", 0) * 100
    beta = info.get("beta")
    debt_to_equity = info.get("debtToEquity")
    margin = info.get("operatingMargins", 0) * 100

    # --- Advanced metrics ---
    peg = info.get("pegRatio")
    pegy = safe_div(pe, eps_growth + div_yield)
    roic = compute_roic(info) * 100 if compute_roic(info) else np.nan

    # ----------------------------
    # Scoring
    # ----------------------------
    score = 0

    # Growth
    if eps_growth > 10: score += 15
    elif eps_growth > 5: score += 8

    if rev_growth > 8: score += 10
    elif rev_growth > 4: score += 5

    # Valuation
    if pegy and pegy < 1: score += 20
    elif pegy and pegy < 1.5: score += 10

    # Quality
    if roic and roic > 12: score += 20
    elif roic and roic > 8: score += 10

    if margin > 20: score += 10
    elif margin > 10: score += 5

    # Risk penalty
    if debt_to_equity and debt_to_equity > 150: score -= 10
    if beta and beta > 1.5: score -= 5

    score = max(0, min(100, score))

    # ----------------------------
    # Report
    # ----------------------------
    print("\n" + "="*60)
    print(f"{info.get('longName')} ({ticker})")
    print(info.get("industry"), "|", info.get("sector"))
    print("-"*60)

    print(f"Price: ${price}")
    print(f"P/E: {pe}")
    print(f"PEGY: {pegy:.2f}" if pegy else "PEGY: N/A")
    print(f"ROIC: {roic:.2f}%" if roic else "ROIC: N/A")
    print(f"Revenue Growth: {rev_growth:.1f}%")
    print(f"EPS Growth: {eps_growth:.1f}%")
    print(f"Operating Margin: {margin:.1f}%")
    print(f"Debt/Equity: {debt_to_equity}")
    print(f"Beta: {beta}")
    print("-"*60)
    print(f"FINAL SCORE: {score}/100 → {verdict(score)}")
    print("="*60)

# ----------------------------
# Run multiple tickers
# ----------------------------

tickers = ["CEG", "VST", "NEE"]  # !!! Change here for tickers

for t in tickers:
    screen_stock(t)


Constellation Energy Corporation (CEG)
Utilities - Independent Power Producers | Utilities
------------------------------------------------------------
Price: $271.14
P/E: 31.022886
PEGY: 0.90
ROIC: 49.36%
Revenue Growth: 0.3%
EPS Growth: -22.5%
Operating Margin: 16.3%
Debt/Equity: 61.51
Beta: 1.136
------------------------------------------------------------
FINAL SCORE: 45/100 → 😐 Neutral – Needs catalyst

Vistra Corp. (VST)
Utilities - Independent Power Producers | Utilities
------------------------------------------------------------
Price: $159.6
P/E: 55.034485
PEGY: -8.47
ROIC: 22.26%
Revenue Growth: -20.9%
EPS Growth: -65.5%
Operating Margin: 21.0%
Debt/Equity: 335.784
Beta: 1.443
------------------------------------------------------------
FINAL SCORE: 40/100 → ❌ Weak – Pass for now

NextEra Energy, Inc. (NEE)
Utilities - Regulated Electric | Utilities
------------------------------------------------------------
Price: $90.83
P/E: 27.276278
PEGY: 0.10
ROIC: 11.94%
Revenue Grow

In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
# ----------------------------
# Helper functions
# ----------------------------
def safe_div(a, b):
    return np.nan if b in [0, None, np.nan] else a / b

def verdict(score):
    if score >= 75:
        return "🔥 Strong – Dig deeper"
    elif score >= 60:
        return "🧐 Interesting – Worth analysis"
    elif score >= 45:
        return "😐 Neutral – Needs catalyst"
    else:
        return "❌ Weak – Pass for now"
# ----------------------------
# Bank Screener
# ----------------------------
def screen_bank(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    # --- Core metrics ---
    price = info.get("currentPrice")
    pe = info.get("trailingPE")
    pb = info.get("priceToBook")
    roe = info.get("returnOnEquity", 0)
    if roe: roe = roe * 100
    
    # FIX: Dividend yield handling
    div_yield_raw = info.get("dividendYield", 0)
    if div_yield_raw:
        # yfinance returns as decimal (0.0395 = 3.95%), but check
        if div_yield_raw < 1:  # It's a decimal
            div_yield = div_yield_raw * 100
        else:  # Already percentage or corrupted data
            div_yield = div_yield_raw
    else:
        div_yield = 0
    
    payout_ratio_raw = info.get("payoutRatio", 0)
    if payout_ratio_raw:
        if payout_ratio_raw < 1:
            payout_ratio = payout_ratio_raw * 100
        else:
            payout_ratio = payout_ratio_raw
    else:
        payout_ratio = 0
    
    eps_growth = info.get("earningsQuarterlyGrowth", 0) * 100
    rev_growth = info.get("revenueGrowth", 0) * 100
    margin = info.get("operatingMargins", 0) * 100
    beta = info.get("beta")
    # ----------------------------
    # Scoring
    # ----------------------------
    score = 0
    # Valuation (30 points) - P/B is critical for banks
    if pb:
        if pb < 1.5: score += 30
        elif pb < 2.0: score += 20
        elif pb < 2.5: score += 12
        elif pb < 3.0: score += 5
    # Profitability (25 points) - ROE matters most
    if roe:
        if roe > 13: score += 25
        elif roe > 10: score += 18
        elif roe > 8: score += 12
        elif roe > 6: score += 5
    # Income (20 points) - Banks are income investments
    if div_yield:
        if div_yield > 5.0: score += 20
        elif div_yield > 4.0: score += 15
        elif div_yield > 3.0: score += 10
        elif div_yield > 2.5: score += 5
    # Growth (15 points) - Lower expectations for banks
    if eps_growth > 15: score += 15
    elif eps_growth > 8: score += 12
    elif eps_growth > 3: score += 8
    elif eps_growth > 0: score += 5
    elif eps_growth > -10: score += 2
    elif eps_growth > -20: score += 1
    if rev_growth > 8: score += 5
    elif rev_growth > 5: score += 4
    elif rev_growth > 3: score += 3
    elif rev_growth > 0: score += 2
    # Risk (10 points)
    if beta:
        if beta < 0.8: score += 10
        elif beta < 1.2: score += 7
    # Penalties
    if payout_ratio and payout_ratio > 90: score -= 5
    if pe and pe > 25: score -= 5
    score = max(0, min(100, score))
    # ----------------------------
    # Report
    # ----------------------------
    print("\n" + "="*60)
    print(f"{info.get('longName')} ({ticker})")
    print(info.get("industry"), "|", info.get("sector"))
    print("-"*60)
    print(f"Price: ${price}")
    print(f"P/E: {pe}")
    print(f"P/B: {pb}")
    print(f"ROE: {roe:.1f}%" if roe else "ROE: N/A")
    print(f"Dividend Yield: {div_yield:.1f}%" if div_yield else "Dividend Yield: N/A")
    print(f"Payout Ratio: {payout_ratio:.1f}%" if payout_ratio else "Payout Ratio: N/A")
    print(f"Revenue Growth: {rev_growth:.1f}%")
    print(f"EPS Growth: {eps_growth:.1f}%")
    print(f"Operating Margin: {margin:.1f}%")
    print(f"Beta: {beta}")
    print("-"*60)
    print(f"FINAL SCORE: {score}/100 → {verdict(score)}")
    print("="*60)
# ----------------------------
# Insurance Screener
# ----------------------------
def screen_insurance(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    # --- Core metrics ---
    price = info.get("currentPrice")
    pe = info.get("trailingPE")
    pb = info.get("priceToBook")
    roe = info.get("returnOnEquity", 0)
    if roe: roe = roe * 100
    
    # FIX: Dividend yield handling
    div_yield_raw = info.get("dividendYield", 0)
    if div_yield_raw:
        if div_yield_raw < 1:
            div_yield = div_yield_raw * 100
        else:
            div_yield = div_yield_raw
    else:
        div_yield = 0
    
    payout_ratio_raw = info.get("payoutRatio", 0)
    if payout_ratio_raw:
        if payout_ratio_raw < 1:
            payout_ratio = payout_ratio_raw * 100
        else:
            payout_ratio = payout_ratio_raw
    else:
        payout_ratio = 0
    
    eps_growth = info.get("earningsQuarterlyGrowth", 0) * 100
    rev_growth = info.get("revenueGrowth", 0) * 100
    operating_margin = info.get("operatingMargins", 0) * 100
    net_margin = info.get("profitMargins", 0) * 100
    beta = info.get("beta")
    # ----------------------------
    # Scoring
    # ----------------------------
    score = 0
    # Valuation (25 points) - P/B important but insurers trade higher
    if pb:
        if pb < 1.2: score += 25
        elif pb < 2.0: score += 18
        elif pb < 3.0: score += 10
    # Profitability (30 points)
    if roe:
        if roe > 12: score += 20
        elif roe > 10: score += 15
        elif roe > 8: score += 10
    if operating_margin:
        if operating_margin > 15: score += 10
        elif operating_margin > 10: score += 7
        elif operating_margin > 5: score += 4
    # Growth (20 points) - Premium growth crucial
    if rev_growth > 10: score += 12
    elif rev_growth > 6: score += 8
    elif rev_growth > 3: score += 5
    if eps_growth > 10: score += 8
    elif eps_growth > 5: score += 5
    elif eps_growth > 0: score += 3
    elif eps_growth > -10: score += 1
    # Income (15 points)
    if div_yield:
        if div_yield > 4: score += 15
        elif div_yield > 3: score += 10
        elif div_yield > 2: score += 6
    # Risk (10 points)
    if beta:
        if beta < 0.9: score += 10
        elif beta < 1.3: score += 6
    # Penalties
    if payout_ratio and payout_ratio > 85: score -= 5
    if pe and pe > 20: score -= 3
    score = max(0, min(100, score))
    # ----------------------------
    # Report
    # ----------------------------
    print("\n" + "="*60)
    print(f"{info.get('longName')} ({ticker})")
    print(info.get("industry"), "|", info.get("sector"))
    print("-"*60)
    print(f"Price: ${price}")
    print(f"P/E: {pe}")
    print(f"P/B: {pb}")
    print(f"ROE: {roe:.1f}%" if roe else "ROE: N/A")
    print(f"Dividend Yield: {div_yield:.1f}%" if div_yield else "Dividend Yield: N/A")
    print(f"Payout Ratio: {payout_ratio:.1f}%" if payout_ratio else "Payout Ratio: N/A")
    print(f"Revenue Growth: {rev_growth:.1f}%")
    print(f"EPS Growth: {eps_growth:.1f}%")
    print(f"Operating Margin: {operating_margin:.1f}%")
    print(f"Net Margin: {net_margin:.1f}%")
    print(f"Beta: {beta}")
    print("-"*60)
    print(f"FINAL SCORE: {score}/100 → {verdict(score)}")
    print("="*60)
# ----------------------------
# Auto-detect and screen
# ----------------------------
def screen_financial(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    industry = info.get('industry', '').lower()
    if 'bank' in industry:
        screen_bank(ticker)
    elif 'insurance' in industry:
        screen_insurance(ticker)
    else:
        # Default to bank screener for other financials
        screen_bank(ticker)
# ----------------------------
# Run multiple tickers
# ----------------------------
tickers = ["NAB.AX", "ANZ.AX", "MQG.AX", "CBA.AX", "WBC.AX"]  # !!! Change here for tickers
for t in tickers:
    screen_financial(t)


National Australia Bank Limited (NAB.AX)
Banks - Diversified | Financial Services
------------------------------------------------------------
Price: $43.0
P/E: 19.457014
P/B: 2.088088
ROE: 10.8%
Dividend Yield: 4.0%
Payout Ratio: 77.0%
Revenue Growth: 3.1%
EPS Growth: -10.4%
Operating Margin: 51.8%
Beta: 0.669
------------------------------------------------------------
FINAL SCORE: 54/100 → 😐 Neutral – Needs catalyst

ANZ Group Holdings Limited (ANZ.AX)
Banks - Diversified | Financial Services
------------------------------------------------------------
Price: $36.505
P/E: 18.625
P/B: 1.5290693
ROE: 8.3%
Dividend Yield: 4.5%
Payout Ratio: 84.5%
Revenue Growth: 6.4%
EPS Growth: -28.1%
Operating Margin: 40.5%
Beta: 0.525
------------------------------------------------------------
FINAL SCORE: 61/100 → 🧐 Interesting – Worth analysis

Macquarie Group Limited (MQG.AX)
Capital Markets | Financial Services
------------------------------------------------------------
Price: $212.94
P/E: 19